# Tutorial: Interconnection Financing Triage

## What is this notebook?

This notebook is a step-by-step tutorial for analyzing EU cross-border electricity interconnection projects and classifying them into **financing tracks**. The goal is to estimate how much of the ~170 bn EUR TYNDP investment pipeline can be market-financed vs. how much needs targeted EU support.

### The three financing tracks

| Track | Description | Financing mix |
|---|---|---|
| **Track 1** | Commercially viable — congestion rents cover costs | Private capital (merchant/hybrid) |
| **Track 2** | Not commercially viable, but TSOs are creditworthy | Regulated tariffs + CBCA |
| **Track 3** | Not commercially viable AND TSO is credit-constrained | CEF grants + EIB loans |

### How to use this notebook

Run each cell in order. Each section explains **what** is being computed and **why**, so you can follow the analytical logic even if you're not familiar with the codebase. All business logic lives in the `grid_financing` Python package — this notebook only orchestrates and displays results.

## Step 0: Setup

We import the `grid_financing` package and configure display settings. The package handles all data loading, calculations, and classification.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots

from grid_financing import (
    build_aggregate_summary,
    build_project_master_table,
    build_sensitivity_cases,
    calculate_project_metrics,
    classify_projects,
    export_outputs,
    manual_source_status,
)
from grid_financing.exports import build_triage_scatter_dataset, build_financing_stack_dataset
from grid_financing.classification import TRACK_1, TRACK_2, TRACK_3

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Use notebook_connected renderer (loads plotly.js from CDN, works in VS Code and Jupyter)
pio.renderers.default = "notebook_connected"

## Step 1: Check data sources

Before loading anything, let's verify which core input files are available. The workflow below only shows sources that already exist in this checkout:

- **TYNDP 2024 workbook** — the main project list, investment-level CAPEX, and CBA welfare scenarios
- **TYNDP 2022 workbook** — older CBA scenarios used as fallback
- **Local hourly prices** — country-level day-ahead prices stored in `data/european_wholesale_electricity_price_data_hourly/all_countries.csv`
- **Manual CSV tables** — TSO credit ratings, project participant CAPEX splits, and border-zone mappings when available

This notebook uses the local hourly CSV already in the repository. It does **not** call an external API, and it does not need the optional manual day-ahead price table for the current workflow. Missing optional sources are handled later through data-quality flags rather than cluttering this setup view.


In [2]:
source_descriptions = {
    "tyndp2022_workbook": ("TYNDP 2022 workbook", "Older CBA scenarios used as fallback"),
    "tyndp2024_workbook": ("TYNDP 2024 workbook", "Main project list, CAPEX, and CBA scenarios"),
    "local_hourly_prices": ("Local hourly prices", "Country-level hourly price series used for the proxy price spreads"),
    "tso_credit_reference": ("TSO credit reference", "TSO ratings, RAB size, and sovereign indicators"),
    "project_participants": ("Project participants", "Participant-level country and CAPEX split records"),
    "project_overrides": ("Project overrides", "Manual overrides for capacities, mappings, or track assignments"),
    "border_zone_map": ("Border-zone map", "Project-to-border mappings used for price processing"),
}

source_status = (
    pd.DataFrame(manual_source_status())
    .query("exists")
    .loc[lambda df: df["source_id"].ne("dayahead_price_inputs")]
    .assign(
        name=lambda df: df["source_id"].map(lambda source_id: source_descriptions[source_id][0]),
        description=lambda df: df["source_id"].map(lambda source_id: source_descriptions[source_id][1]),
    )
    [["name", "description", "variant"]]
    .rename(columns={"variant": "location"})
    .reset_index(drop=True)
)

source_status


,name,description,location
0,TYNDP 2022 workbook,Older CBA scenarios used as fallback,legacy
1,TYNDP 2024 workbook,"Main project list, CAPEX, and CBA scenarios",legacy
2,Local hourly prices,Country-level hourly price series used for the...,canonical
3,TSO credit reference,"TSO ratings, RAB size, and sovereign indicators",canonical
4,Project participants,Participant-level country and CAPEX split records,canonical
5,Border-zone map,Project-to-border mappings used for price proc...,canonical


## Step 2: Build the project master table

This is the core data assembly step. It merges multiple sources into one row per cross-border project in the base case:

1. **Load projects** from the TYNDP 2024 `Trans.Projects` sheet (filters to cross-border only)
2. **Aggregate CAPEX** from `Trans.Investments` (sums investment-level costs per project)
3. **Merge CBA welfare values** from 2024 and 2022 scenario tabs
4. **Apply manual overrides** for missing transfer capacities and country mappings
5. **Attach credit data** — TSO ratings, RAB sizes, sovereign fiscal indicators, cohesion status
6. **Preprocess hourly prices** — in development mode, build border spreads from overlapping hourly observations in the local CSV and keep the latest full year for the base case
7. **Flag data-quality issues** — missing CAPEX, missing prices, unresolved multi-country projects, etc.

The year-by-year hourly preprocessing can also expand the table into multiple full-year price scenarios for sensitivity work, for example `proxy_2020` through `proxy_2025`.


In [3]:
master = build_project_master_table(development_mode=True)

print(f"Cross-border projects loaded: {master['project_id'].nunique()}")
print(f"Project rows in base case: {len(master)}")
print(f"Projects with data-quality flags: {master['has_data_quality_issue'].sum()}")
print(f"Price input mode: {master['price_input_mode'].iloc[0]}")
print()

preview = master[[
    "project_id", "project_name", "countries", "status",
    "capex_meur", "capacity_mw",
    "price_scenario", "data_year", "hourly_observation_count",
    "avg_price_diff_eur_per_mwh", "data_quality_flags",
]].head(10)

data_issue_summary = (
    master.loc[master["has_data_quality_issue"], "data_quality_flags"]
    .dropna()
    .str.split(";")
    .explode()
    .value_counts()
    .rename_axis("issue")
    .reset_index(name="project_count")
)

print("Base-case preview:")
display(preview)
print("Data issue summary:")
display(data_issue_summary)


Cross-border projects loaded: 100
Project rows in base case: 100
Projects with data-quality flags: 47
Price input mode: development-local-proxy

Base-case preview:


,project_id,project_name,countries,status,capex_meur,capacity_mw,price_scenario,data_year,hourly_observation_count,avg_price_diff_eur_per_mwh,data_quality_flags
0,4.0,Interconnection Portugal-Spain,ES ; PT,Construction,111.18,1500.0,proxy_2025,2025.0,8760.0,1.347471,<NA>
1,16.0,Biscay Gulf,ES ; FR,Construction,3100.00,2200.0,proxy_2025,2025.0,8760.0,20.325945,<NA>
2,28.0,Italy-Montenegro,IT ; ME,Construction,424.00,600.0,proxy_2024,2024.0,8784.0,19.686683,missing_credit_reference
3,29.0,Italy-Tunisia,IT ; TN,Construction,850.00,600.0,historical_proxy,NaN,NaN,NaN,missing_credit_reference;missing_price_inputs
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,BE ; DE ; LU,Under Consideration,210.00,500.0,proxy_2025,2025.0,8760.0,12.475568,missing_credit_reference
5,47.0,Westtirol (AT) - Vöhringen (DE),AT ; DE,Planning,10.30,600.0,proxy_2025,2025.0,8760.0,13.041811,<NA>
6,81.0,North South Interconnector,GB ; IE,Permitting,363.00,950.0,proxy_2025,2025.0,8760.0,31.960582,missing_credit_reference
7,94.0,GerPol Improvements,DE ; PL,Construction,110.80,1500.0,proxy_2025,2025.0,8760.0,19.908811,<NA>
8,107.0,Celtic Interconnector,FR ; IE,Construction,1623.00,700.0,proxy_2025,2025.0,8760.0,58.082207,<NA>
9,111.0,Aurora line (3rd AC Finland-Sweden north),FI ; SE,Construction,270.00,900.0,proxy_2025,2025.0,8760.0,21.458082,<NA>


Data issue summary:


,issue,project_count
0,missing_credit_reference,45
1,missing_price_inputs,20
2,missing_transfer_capacity,6


### Step 2.1: Explore the pipeline — CAPEX by project status

Before running the triage, let's understand what's in the TYNDP pipeline. The 100 cross-border projects span four development stages:

- **Under Consideration** — earliest stage, no permitting yet
- **Planning** — in planning but not yet in permitting
- **Permitting** — actively going through permitting
- **Construction** — already under construction, likely already financed

**Projects "Under Construction" (~14 bn EUR) are likely already financed**, so the remaining ~156 bn EUR is the relevant financing gap — consistent with the ~150 bn figure cited in the policy brief.

In [4]:
# CAPEX breakdown by project development status
status_summary = (
    master.groupby("status")
    .agg(
        project_count=("project_id", "nunique"),
        total_capex_meur=("capex_meur", "sum"),
    )
    .sort_values("total_capex_meur", ascending=False)
)
status_summary["total_capex_beur"] = status_summary["total_capex_meur"] / 1000
status_summary["share_pct"] = (
    status_summary["total_capex_meur"] / status_summary["total_capex_meur"].sum() * 100
).round(1)

print(f"Total pipeline: {status_summary['total_capex_meur'].sum()/1000:.1f} bn EUR "
      f"across {status_summary['project_count'].sum()} projects")
print(f"Excluding Construction: "
      f"{(status_summary['total_capex_meur'].sum() - status_summary.loc['Construction', 'total_capex_meur'])/1000:.1f} bn EUR")
print()
status_summary[["project_count", "total_capex_beur", "share_pct"]]

Total pipeline: 169.9 bn EUR across 100 projects
Excluding Construction: 156.1 bn EUR



,project_count,total_capex_beur,share_pct
status,,,
Under Consideration,40,123.214084,72.5
Planning,29,23.561613,13.9
Construction,12,13.790713,8.1
Permitting,19,9.330541,5.5


In [5]:
# Visualize CAPEX by status
fig_status = px.bar(
    status_summary.reset_index(),
    x="status",
    y="total_capex_beur",
    text="project_count",
    title="CAPEX Pipeline by Project Status (bn EUR)",
    labels={"total_capex_beur": "Total CAPEX (bn EUR)", "status": "Development status"},
    color="status",
    category_orders={"status": ["Under Consideration", "Planning", "Permitting", "Construction"]},
)
fig_status.update_traces(texttemplate="%{text} projects", textposition="outside")
fig_status.update_layout(showlegend=False, height=450)
fig_status.show()

### Step 2.2: CAPEX vs capacity across all projects

A single scatter plot makes it easier to see the full project pipeline at once. The x-axis shows transfer capacity in GW and the y-axis shows project CAPEX in bn EUR.

This highlights whether the pipeline is dominated by a few very large, high-capex links or by many smaller reinforcements.


In [6]:
scatter_projects = master[[
    "project_id", "project_name", "countries", "status", "capex_meur", "capacity_mw"
]].copy()
scatter_projects["capacity_gw"] = scatter_projects["capacity_mw"] / 1000
scatter_projects["capex_beur"] = scatter_projects["capex_meur"] / 1000

fig_pipeline = px.scatter(
    scatter_projects,
    x="capacity_gw",
    y="capex_beur",
    color="status",
    hover_name="project_name",
    hover_data={
        "project_id": True,
        "countries": True,
        "capacity_gw": ":.2f",
        "capex_beur": ":.2f",
        "capacity_mw": False,
        "capex_meur": False,
    },
    title="Project Pipeline: Capacity vs CAPEX",
    labels={
        "capacity_gw": "Transfer capacity (GW)",
        "capex_beur": "CAPEX (bn EUR)",
        "status": "Development status",
    },
    category_orders={"status": ["Under Consideration", "Planning", "Permitting", "Construction"]},
)
fig_pipeline.update_traces(marker={"size": 11, "line": {"width": 0.5, "color": "white"}})
fig_pipeline.update_layout(height=520)
fig_pipeline.show()


## Step 3: Calculate financing metrics

Now we compute the ratios that drive the triage classification. Here's what each metric means:

| Metric | Formula | Interpretation |
|---|---|---|
| **Capital Recovery Factor** | `r / (1 - (1+r)^(-n))` | Annual repayment as fraction of initial cost (r=5%, n=25yr -> CRF≈0.071) |
| **Annualized CAPEX** | `CAPEX × CRF` | Yearly cost the project must cover to break even |
| **Congestion rent** | `sum_t abs(price_a,t - price_b,t) × capacity × utilization / 1M` | Revenue estimated from the **absolute hourly spread sum**, not from the signed sum |
| **Commercial ratio** | `congestion_rent / annualized_CAPEX` | >1.0 means the project covers its annualized investment cost |
| **Social BCR** | `ΔSEW / annualized_CAPEX` | Welfare benefit per euro of cost (from TYNDP CBA) |
| **Credit constraint score** | `project_CAPEX_share / TSO_RAB` | How large the project is relative to the sponsoring TSO's balance sheet |

The implementation uses `abs(price_a - price_b)` hour by hour and then sums across the year before multiplying by capacity and utilization.

For the base case, we annualize CAPEX over **25 years**. That is a pragmatic merchant-finance benchmark: companies generally want the project to cover its investment over roughly **25 years**, not over the full technical lifetime of the asset.

**Caveat:** this congestion-rent estimate is a **majorant**. It is an upper-bound style proxy because it assumes the link can monetize the full absolute hourly spread at the chosen utilization factor, without losses, outages, ramping constraints, directional congestion limits, or other market-frictions.


In [7]:
metrics = calculate_project_metrics(master)

print(f"CRF at 5% / 25yr: {metrics['capital_recovery_factor'].iloc[0]:.4f}")
print(f"Projects with computable commercial ratio: {metrics['commercial_ratio'].notna().sum()} / {len(metrics)}")
print()

metrics[[
    "project_id", "project_name", "price_scenario", "data_year",
    "hourly_observation_count", "congestion_rent_basis",
    "annualized_capex_meur_per_year",
    "estimated_congestion_rent_meur_per_year",
    "commercial_ratio", "social_bcr",
    "credit_constraint_score", "credit_constrained",
]].head(15)


CRF at 5% / 25yr: 0.0710
Projects with computable commercial ratio: 76 / 100



,project_id,project_name,price_scenario,data_year,hourly_observation_count,congestion_rent_basis,annualized_capex_meur_per_year,estimated_congestion_rent_meur_per_year,commercial_ratio,social_bcr,credit_constraint_score,credit_constrained
0,4.0,Interconnection Portugal-Spain,proxy_2025,2025.0,8760.0,hourly_price_sum,7.888494,10.623465,1.346704,3.295940,0.015883,False
1,16.0,Biscay Gulf,proxy_2025,2025.0,8760.0,hourly_price_sum,219.952618,235.03297,1.068562,1.900409,0.166667,True
2,28.0,Italy-Montenegro,proxy_2024,2024.0,8784.0,hourly_price_sum,30.083842,62.254015,2.069351,0.199443,0.009422,False
3,29.0,Italy-Tunisia,historical_proxy,NaN,NaN,hourly_price_sum,60.309589,NaN,NaN,0.829056,0.018889,False
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,proxy_2025,2025.0,8760.0,hourly_price_sum,14.900016,32.785794,2.200386,1.476508,NaN,False
5,47.0,Westtirol (AT) - Vöhringen (DE),proxy_2025,2025.0,8760.0,hourly_price_sum,0.730810,41.128654,56.278152,47.892045,0.001717,True
6,81.0,North South Interconnector,proxy_2025,2025.0,8760.0,hourly_price_sum,25.755742,159.585579,6.196117,0.155305,0.051857,True
7,94.0,GerPol Improvements,proxy_2025,2025.0,8760.0,hourly_price_sum,7.861532,156.961062,19.965709,17.808233,0.011080,True
8,107.0,Celtic Interconnector,proxy_2025,2025.0,8760.0,hourly_price_sum,115.155838,213.696055,1.855712,2.110184,0.231857,True
9,111.0,Aurora line (3rd AC Finland-Sweden north),proxy_2025,2025.0,8760.0,hourly_price_sum,19.157163,101.505312,5.298556,2.401191,0.038571,True


### Step 3.1: Congestion rent heatmap by country pair (2015-2021)

To see how congestion-rent potential moves over time, we rebuild the yearly price scenarios from 2015 to 2021 and aggregate the estimated congestion rent by country pair.

The heatmaps below use only project-year rows with a full overlapping hourly year. Country pairs are mirrored so each bilateral corridor appears symmetrically in the matrix. Values are in MEUR per year.


In [8]:
heatmap_years = range(2015, 2022)
yearly_master = build_project_master_table(development_mode=True, price_years=heatmap_years)
yearly_master = yearly_master[
    yearly_master["hourly_observation_count"].notna()
    & yearly_master["country_a"].notna()
    & yearly_master["country_b"].notna()
].copy()
yearly_metrics = calculate_project_metrics(yearly_master)

country_pairs = yearly_metrics.apply(
    lambda row: tuple(sorted((str(row["country_a"]), str(row["country_b"])))),
    axis=1,
)
yearly_metrics[["country_row", "country_col"]] = pd.DataFrame(country_pairs.tolist(), index=yearly_metrics.index)

pair_rents = (
    yearly_metrics.groupby(["data_year", "country_row", "country_col"], dropna=False)["estimated_congestion_rent_meur_per_year"]
    .sum()
    .reset_index()
)

mirrored_pair_rents = pd.concat([
    pair_rents,
    pair_rents.rename(columns={"country_row": "country_col", "country_col": "country_row"}),
], ignore_index=True).drop_duplicates(["data_year", "country_row", "country_col"])

countries = sorted(set(mirrored_pair_rents["country_row"]).union(mirrored_pair_rents["country_col"]))
zmax = mirrored_pair_rents["estimated_congestion_rent_meur_per_year"].max()

fig_heatmap = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=[str(year) for year in heatmap_years],
    horizontal_spacing=0.04,
    vertical_spacing=0.08,
)

for index, year in enumerate(heatmap_years):
    row = index // 3 + 1
    col = index % 3 + 1
    matrix = (
        mirrored_pair_rents.loc[mirrored_pair_rents["data_year"] == year]
        .pivot(index="country_row", columns="country_col", values="estimated_congestion_rent_meur_per_year")
        .reindex(index=countries, columns=countries, fill_value=0)
    )
    fig_heatmap.add_trace(
        go.Heatmap(
            z=matrix.values,
            x=matrix.columns,
            y=matrix.index,
            coloraxis="coloraxis",
            zmin=0,
            zmax=zmax,
            hovertemplate="Row country: %{y}<br>Column country: %{x}<br>Congestion rent: %{z:.1f} MEUR/year<extra></extra>",
        ),
        row=row,
        col=col,
    )

fig_heatmap.update_layout(
    height=1100,
    title="Congestion rent by country pair and year (MEUR/year)",
    coloraxis={"colorscale": "YlOrRd", "colorbar": {"title": "MEUR/year"}},
)
fig_heatmap.show()


### Step 3.2: What is TSO RAB and how does the credit-constraint metric work?

**TSO RAB** means the **regulated asset base** of the transmission system operator. In practice, it is a proxy for the size of the regulated balance sheet on which the TSO can earn an allowed return.

The credit-constraint score asks a simple question for each project side: **how large is that side of the project compared with the sponsoring TSO's RAB?**

`credit constraint score = project CAPEX share / TSO RAB`

A higher score means the project is large relative to the TSO's balance sheet. In the base case, a score above **0.15** means the project side is more than **15%** of that TSO's RAB, which is our main quantitative flag for a possible financing constraint.

For transparency, the notebook uses the **country-side mapping attached to each project row**: side **A** uses `country_a_participant` and `tso_a_name_reference` where available, while side **B** uses `country_b_participant` and `tso_b_name_reference`. If participant-level fields are missing, the pipeline falls back to the broader border-side country mapping.


In [9]:
credit_sides = pd.concat([
    metrics[[
        "project_id", "project_name", "countries", "status",
        "country_a", "country_a_participant",
        "project_capex_share_a_meur", "tso_a_rab_beur", "credit_constraint_score_a",
        "tso_a_name_reference",
    ]].rename(columns={
        "country_a": "border_country",
        "country_a_participant": "participant_country",
        "project_capex_share_a_meur": "project_capex_share_meur",
        "tso_a_rab_beur": "tso_rab_beur",
        "credit_constraint_score_a": "credit_constraint_score_side",
        "tso_a_name_reference": "tso_name",
    }).assign(side="A"),
    metrics[[
        "project_id", "project_name", "countries", "status",
        "country_b", "country_b_participant",
        "project_capex_share_b_meur", "tso_b_rab_beur", "credit_constraint_score_b",
        "tso_b_name_reference",
    ]].rename(columns={
        "country_b": "border_country",
        "country_b_participant": "participant_country",
        "project_capex_share_b_meur": "project_capex_share_meur",
        "tso_b_rab_beur": "tso_rab_beur",
        "credit_constraint_score_b": "credit_constraint_score_side",
        "tso_b_name_reference": "tso_name",
    }).assign(side="B"),
], ignore_index=True)

credit_sides["country_side"] = credit_sides["participant_country"].fillna(credit_sides["border_country"])
credit_sides = credit_sides[
    credit_sides["project_capex_share_meur"].notna() & credit_sides["tso_rab_beur"].notna()
].copy()
credit_sides["threshold_exceeded"] = credit_sides["credit_constraint_score_side"] > 0.15

credit_side_mapping = credit_sides[[
    "project_name", "countries", "side", "country_side", "tso_name",
    "tso_rab_beur", "project_capex_share_meur", "credit_constraint_score_side",
]].rename(columns={"credit_constraint_score_side": "credit_constraint_score"}).head(12)
credit_side_mapping

fig_credit = px.scatter(
    credit_sides,
    x="tso_rab_beur",
    y="project_capex_share_meur",
    color="threshold_exceeded",
    hover_name="project_name",
    hover_data={
        "countries": True,
        "side": True,
        "country_side": True,
        "tso_name": True,
        "tso_rab_beur": ":.1f",
        "project_capex_share_meur": ":.1f",
        "credit_constraint_score_side": ":.3f",
    },
    title="Project CAPEX share vs TSO RAB by project side",
    labels={
        "tso_rab_beur": "TSO RAB (bn EUR)",
        "project_capex_share_meur": "Project CAPEX share (MEUR)",
        "threshold_exceeded": "Score > 0.15",
    },
    color_discrete_map={False: "#4c78a8", True: "#d62728"},
)
xmax = credit_sides["tso_rab_beur"].max()
fig_credit.add_scatter(
    x=[0, xmax],
    y=[0, 150 * xmax],
    mode="lines",
    name="15% threshold",
    line={"dash": "dash", "color": "gray"},
)
fig_credit.update_layout(height=520)
fig_credit.show()

most_constrained_sides = credit_sides.nlargest(12, "credit_constraint_score_side")[[
    "project_name", "countries", "side", "country_side", "tso_name",
    "tso_rab_beur", "project_capex_share_meur", "credit_constraint_score_side",
]].rename(columns={"credit_constraint_score_side": "credit_constraint_score"})
most_constrained_sides


,project_name,countries,side,country_side,tso_name,tso_rab_beur,project_capex_share_meur,credit_constraint_score
95,NaN,MT ; TN,A,MT,Enemalta plc,0.2,744.50,3.722500
92,NaN,HU ; RO,A,HU,MAVIR ZRt,1.5,3642.00,2.428000
192,NaN,HU ; RO,B,RO,Transelectrica SA,1.5,3642.00,2.428000
196,NaN,DE ; GR,B,GR,ADMIE/IPTO (Independent Power Transmission Ope...,2.0,4050.00,2.025000
65,Off-shore Wind Park in Latvia and Estonia - EL...,EE ; LV,A,EE,Elering AS,0.5,612.50,1.225000
67,Estlink 3,EE ; FI,A,EE,Elering AS,0.5,533.00,1.066000
166,Offshore Hybrid HVDC Interconnector BEDK,BE ; DK,B,DK,Energinet,5.0,5046.00,1.009200
155,GREGY Green Energy Interconnector | Purifying ...,EG ; GR,B,GR,ADMIE/IPTO (Independent Power Transmission Ope...,2.0,1784.50,0.892250
165,Off-shore Wind Park in Latvia and Estonia - EL...,EE ; LV,B,LV,AST (AS Augstsprieguma tikls),0.8,612.50,0.765625
164,Malta-Italy Cable Link No.2,IT ; MT,B,MT,Enemalta plc,0.2,142.75,0.713750


## Step 4: Classify projects into financing tracks

The classification logic is straightforward:

1. **Commercial ratio > 1.0?** → **Track 1** (market-financed). Congestion rents are high enough to cover the annualized investment cost under a merchant or hybrid structure.
2. **Not commercially viable, but TSO is creditworthy?** → **Track 2** (regulated + CBCA). The TSO can finance through regulated tariffs with a Cross-Border Cost Allocation agreement.
3. **Not commercially viable AND credit-constrained?** → **Track 3** (needs EU support). The TSO's balance sheet is too small relative to project cost, or the sovereign is fiscally stressed. These need CEF grants + EIB loans.
4. **Missing data?** → **Unclassified**. Projects without price data or transfer capacity can't be evaluated.

A project is "credit-constrained" if **any** of these hold for **either side**:
- Credit constraint score > 0.15 (project is >15% of TSO RAB)
- TSO has sub-investment-grade rating
- Sovereign has debt >100% GDP AND deficit >3% GDP

After classification, the financing stack is estimated per project. Track 3 CEF grant rates default to 85% for cohesion-country sides and 50% otherwise.

In [10]:
classified = classify_projects(metrics)

print("Track distribution:")
print(classified["financing_track"].value_counts().to_string())
print()

classified[[
    "project_id", "project_name", "countries",
    "capex_meur", "commercial_ratio", "social_bcr",
    "credit_constraint_score", "financing_track",
    "estimated_cef_grant_meur", "data_quality_flags",
]].head(15)

Track distribution:
financing_track
Track 1: Market-financed (merchant/hybrid)           60
Unclassified: insufficient data                      24
Track 3: Credit-constrained - targeted EU support    11
Track 2: Regulated + CBCA                             5



,project_id,project_name,countries,capex_meur,commercial_ratio,social_bcr,credit_constraint_score,financing_track,estimated_cef_grant_meur,data_quality_flags
0,4.0,Interconnection Portugal-Spain,ES ; PT,111.18,1.346704,3.295940,0.015883,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
1,16.0,Biscay Gulf,ES ; FR,3100.00,1.068562,1.900409,0.166667,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
2,28.0,Italy-Montenegro,IT ; ME,424.00,2.069351,0.199443,0.009422,Track 1: Market-financed (merchant/hybrid),0.0,missing_credit_reference
3,29.0,Italy-Tunisia,IT ; TN,850.00,NaN,0.829056,0.018889,Unclassified: insufficient data,0.0,missing_credit_reference;missing_price_inputs
4,40.0,Belgium-Luxembourg-Germany: long-term perspective,BE ; DE ; LU,210.00,2.200386,1.476508,NaN,Track 1: Market-financed (merchant/hybrid),0.0,missing_credit_reference
5,47.0,Westtirol (AT) - Vöhringen (DE),AT ; DE,10.30,56.278152,47.892045,0.001717,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
6,81.0,North South Interconnector,GB ; IE,363.00,6.196117,0.155305,0.051857,Track 1: Market-financed (merchant/hybrid),0.0,missing_credit_reference
7,94.0,GerPol Improvements,DE ; PL,110.80,19.965709,17.808233,0.011080,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
8,107.0,Celtic Interconnector,FR ; IE,1623.00,1.855712,2.110184,0.231857,Track 1: Market-financed (merchant/hybrid),0.0,<NA>
9,111.0,Aurora line (3rd AC Finland-Sweden north),FI ; SE,270.00,5.298556,2.401191,0.038571,Track 1: Market-financed (merchant/hybrid),0.0,<NA>


## Step 5: Visualize the triage results

### 5.1 Scatter plot — the core policy chart

This is the centrepiece visualization for the policy brief. Each dot is a project:

- **X-axis:** commercial viability ratio. Right of the dashed line at 1.0 = congestion rents cover annualized project cost.
- **Y-axis:** credit constraint score. Above the dashed line at 0.15 = TSO balance sheet is strained.
- **Dot size:** total CAPEX.
- **Color:** financing track.

The quadrants map directly to tracks:
- **Right half** → Track 1 (commercially viable, any credit profile)
- **Bottom-left** → Track 2 (not viable, but creditworthy TSO)
- **Top-left** → Track 3 (not viable AND credit-constrained)

In [11]:
scatter_df = build_triage_scatter_dataset(classified)

fig_scatter = px.scatter(
    scatter_df,
    x="commercial_ratio",
    y="credit_constraint_score",
    size="capex_meur",
    color="financing_track",
    hover_name="project_name",
    title="Financing Triage: Commercial Viability vs. Credit Constraint",
    labels={
        "commercial_ratio": "Commercial viability ratio",
        "credit_constraint_score": "Credit constraint score (CAPEX share / TSO RAB)",
    },
)
fig_scatter.add_vline(x=1.0, line_dash="dash", line_color="gray",
                       annotation_text="Commercial threshold (1.0)")
fig_scatter.add_hline(y=0.15, line_dash="dash", line_color="gray",
                       annotation_text="Credit constraint threshold (0.15)")
fig_scatter.update_layout(height=600)
fig_scatter.show()

### 5.2 Aggregate financing needs

How does the ~170 bn EUR pipeline break down by financing track and source?

The key policy insight: **if merchant financing works for Track 1 projects, CEF resources can be concentrated on the credit-constrained projects (Track 3) that need them most.**

In [12]:
summary = build_aggregate_summary(classified)
summary[[
    "financing_track", "project_count",
    "total_capex_beur", "total_cef_beur", "total_eib_beur",
    "total_private_beur", "total_tso_beur",
]]

,financing_track,project_count,total_capex_beur,total_cef_beur,total_eib_beur,total_private_beur,total_tso_beur
0,Track 1: Market-financed (merchant/hybrid),60,43.547619,0.000000,0.000000,43.547619,0.000000
1,Track 2: Regulated + CBCA,5,18.785000,5.635500,4.696250,0.000000,8.453250
2,Track 3: Credit-constrained - targeted EU support,11,33.828190,22.864235,6.765638,0.000000,5.048337
3,Unclassified: insufficient data,24,73.736142,0.000000,0.000000,0.000000,0.000000
4,Total,100,169.896951,28.499735,11.461888,43.547619,13.501587


### 5.3 Financing stack chart

Stacked bar showing the financing breakdown per track. Compare the CEF grant total with the proposed CEF-E budget (~17 bn EUR for electricity).

In [13]:
stack_df = build_financing_stack_dataset(summary)

fig_stack = px.bar(
    stack_df,
    x="financing_track",
    y="value_beur",
    color="financing_component",
    barmode="stack",
    title="Aggregate Financing Stack by Track (bn EUR)",
    labels={"value_beur": "Financing (bn EUR)", "financing_track": ""},
    category_orders={"financing_track": [TRACK_1, TRACK_2, TRACK_3]},
)
fig_stack.update_layout(height=500)
fig_stack.show()

## Step 6: Sensitivity analysis

The thresholds and assumptions above are inherently uncertain. Here we combine financing assumptions with full-year hourly price windows from 2020 to 2025:

- **Price years:** 2020, 2021, 2022, 2023, 2024, 2025 — changes congestion rents using observed full-year hourly spreads from the local dataset
- **Discount rates:** 4%, 5% (base), 6% — changes annualized CAPEX and therefore all ratios
- **Utilization factors:** 50%, 60% (base), 70% — changes congestion rent estimates
- **Credit thresholds:** 10%, 15% (base), 20% — shifts projects between Track 2 and 3

Only project-year rows with a full overlapping hourly year are kept in this sensitivity table. That yields up to `6 × 3 × 3 × 3 = 162` combinations per project where full-year price data exist.


In [14]:
price_years = range(2020, 2026)
sensitivity_master = build_project_master_table(development_mode=True, price_years=price_years)
sensitivity_master = sensitivity_master[sensitivity_master["hourly_observation_count"].notna()].copy()
sensitivity = build_sensitivity_cases(sensitivity_master, price_scenarios=tuple(f"proxy_{year}" for year in price_years))

print(f"Sensitivity source rows with full-year hourly data: {len(sensitivity_master)}")
print(f"Sensitivity cases: {sensitivity['sensitivity_case'].nunique()}")
print(f"Sensitivity output rows: {len(sensitivity)}")
print()

sensitivity[[
    "project_id", "project_name", "price_scenario", "data_year",
    "hourly_observation_count", "commercial_ratio",
    "credit_constraint_score", "credit_constrained",
]].head(15)


Sensitivity source rows with full-year hourly data: 456
Sensitivity cases: 162
Sensitivity output rows: 12312



,project_id,project_name,price_scenario,data_year,hourly_observation_count,commercial_ratio,credit_constraint_score,credit_constrained
0,4.0,Interconnection Portugal-Spain,proxy_2020,2020,8784.0,0.111437,0.015883,False
1,16.0,Biscay Gulf,proxy_2020,2020,8784.0,0.284907,0.166667,True
2,40.0,Belgium-Luxembourg-Germany: long-term perspective,proxy_2020,2020,8784.0,0.690155,NaN,False
3,47.0,Westtirol (AT) - Vöhringen (DE),proxy_2020,2020,8784.0,13.141867,0.001717,True
4,81.0,North South Interconnector,proxy_2020,2020,8784.0,1.929884,0.051857,True
5,94.0,GerPol Improvements,proxy_2020,2020,8784.0,15.482839,0.011080,True
6,107.0,Celtic Interconnector,proxy_2020,2020,8784.0,0.390237,0.231857,True
7,111.0,Aurora line (3rd AC Finland-Sweden north),proxy_2020,2020,8784.0,1.771093,0.038571,True
8,121.0,Nautilus: multi-purpose interconnector Belgium...,proxy_2020,2020,8784.0,0.807583,0.027027,False
9,124.0,NordBalt phase 2,proxy_2020,2020,8784.0,0.0,NaN,False


## Step 7: Export results

Write all outputs to disk for the policy brief:

- `data/processed/project_master_table.csv` — full project-level table with all metrics and flags
- `data/processed/aggregate_summary.csv` — per-track totals
- `outputs/tables/` — Excel versions for reviewers
- `outputs/charts/` — interactive HTML scatter and stacked-bar charts

All exported tables include assumptions, source provenance, and data-quality flags so reviewers can trace any metric back to its source.

In [15]:
exports = export_outputs(classified)

print("Exported files:")
for name, path in exports.items():
    print(f"  {name}: {path}")

Exported files:
  project_master_table_csv: /Users/lucas/Workspace/PolicyBrief/GridFinancing/data/processed/project_master_table.csv
  aggregate_summary_csv: /Users/lucas/Workspace/PolicyBrief/GridFinancing/data/processed/aggregate_summary.csv
  project_level_results_xlsx: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/tables/project_level_results.xlsx
  aggregate_summary_xlsx: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/tables/aggregate_summary.xlsx
  triage_scatter_html: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/charts/triage_scatter.html
  aggregate_financing_stack_html: /Users/lucas/Workspace/PolicyBrief/GridFinancing/outputs/charts/aggregate_financing_stack.html
